In [ ]:
import os
import re
import json
import pickle
import numpy as np
import pandas as pd
from rich.pretty import pprint
import matplotlib.pyplot as plt
from collections import defaultdict
from more_itertools import unique_justseen
from aymurai.database.utils import text_to_uuid
from aymurai.api.endpoints.routers.misc.document_extract import extraction

docs2analize = ['2','3','4','6','7','8']
IS_LLM_FILE = True # or False
LLM_PREDICTION_FILE = "predictions_openai-alldocsv1.pkl"
AYMURAI_JSONS_PATH = '/Users/sofi/Desktop/collectiveai/projects/AymurAI/' #'/Users/sofi/Desktop/collectiveAI/projects/AymurAI' # '/Users/sofi/Desktop/collectiveai/projects/AymurAI/'
DOCS_PATH = '/Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/resources/data/sample/' #'/Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/resources/data/sample/' 

In [ ]:
def get_paragraph_predictions(row,predictions):
    preds = predictions.get(row['name'], [])
    return [
        p for p in preds
        if p['start_char'] >= (row['start_char']) and (p['end_char'] <= row['end_char'])
    ]

def take_start_end_paragraphs(paragraphs):
    # Create start and end character positions
    start_end_chars = []
    current_pos = 0

    for i, paragraph in enumerate(paragraphs):
        start_char = current_pos
        end_char = start_char + len(paragraph)
        start_end_chars.append(
            {
                "paragraph_position": i,
                "text": paragraph,
                "paragraph_id": str(text_to_uuid(paragraph)).replace("-", ""),
                "start_char": start_char,
                "end_char": end_char,
            }
        )
        # +1 for the newline character between paragraphs (except after the last one)
        current_pos = end_char + 1

    return start_end_chars

def constract_paragraph(document):
    paragraphs = [line.strip() for line in document.split("\n") if line.strip()]
    paragraphs = [re.sub(r"\s{2,}", " ", line) for line in paragraphs]
    paragraphs = list(unique_justseen(paragraphs))
    return paragraphs


# Prepare dataframe

## Join databased .json (NER)

In [ ]:
json_files = [f for f in os.listdir(AYMURAI_JSONS_PATH) if f.endswith('.json')]
para_files = [f for f in json_files if f.startswith('anonymization_paragraph')]
para_df = pd.concat([pd.read_json(AYMURAI_JSONS_PATH + fname) for fname in para_files], ignore_index=True)

doc_files = [f for f in json_files if f.startswith('anonymization_document') and not 'paragraph' in f]
doc_df = pd.concat([pd.read_json(AYMURAI_JSONS_PATH + fname) for fname in doc_files], ignore_index=True)

doc_para_files = [f for f in json_files if f.startswith('anonymization_document_paragraph')]
doc_para_df = pd.concat([pd.read_json(AYMURAI_JSONS_PATH + fname) for fname in doc_para_files], ignore_index=True)

doc_df = doc_df.copy()
doc_para_df = doc_para_df.copy()
doc_df['id'] = doc_df['id'].astype(str)
doc_para_df['document_id'] = doc_para_df['document_id'].astype(str)

merged_df = pd.merge(
    doc_para_df,
    doc_df[['id', 'created_at', 'name']],
    left_on='document_id',
    right_on='id',
    how='left'
).rename(columns={'id_x': 'id'}).drop(columns=['id_y'])

df = pd.merge(
    para_df,
    merged_df,
    left_on='id',
    right_on='paragraph_id',
    how='left'
)

df = df[df['name'].fillna('').str.startswith('document') & (df['name']!= 'doc')]
df = df.rename(columns={'prediction':'NER_prediction'})
df.head(2)

## Load validation .docx

In [ ]:
docs2analize = ['2','3','4','6','7','8']
docs_file = [f for f in os.listdir(DOCS_PATH) if ('.docx' in f) and (len(set(docs2analize)&set(f))==1) ]

documents = {}
doc_paragraphs = {}
doc_start_end_chars = {}
joined_texts = {}
for d in docs_file:
    print(d)
    path = DOCS_PATH + d
    # Extract document
    document = extraction(path)
    # Construct paragraphs
    paragraphs = constract_paragraph(document)
    start_end_chars = take_start_end_paragraphs(paragraphs)
    documents[d] = document
    doc_paragraphs[d] = paragraphs
    joined_text = "\n".join(paragraphs)
    joined_texts[d] = joined_text
    doc_start_end_chars[d] = start_end_chars
    print(start_end_chars)
    print('\n')

In [ ]:
dfs = []
for doc, start_end_chars in doc_start_end_chars.items():
    df_chars = pd.DataFrame(start_end_chars)
    df_chars['doc'] = doc
    dfs.append(df_chars)
all_df_chars = pd.concat(dfs, ignore_index=True)

all_df_chars.head()

In [ ]:
main_df = df.merge(
    all_df_chars,
    left_on=['paragraph_id', 'name'],
    right_on=['paragraph_id', 'doc'],
    how='inner'
)
#main_df.to_csv('documents-02-08-sin05-conOpenAI.csv', index=False)
main_df.head()

In [ ]:
main_df[main_df['name']!=main_df['doc']]

## Load LLM prediction database .pkl

In [ ]:
if IS_LLM_FILE:
    try:
        with open(LLM_PREDICTION_FILE, "rb") as file:
            predictions = pickle.load(file)
    except Exception as e:
        print(f'There is no file named {LLM_PREDICTION_FILE}, please redefine LLM_PREDICTION_FILE variable')
else:
    print('There is no file of predictions')

In [ ]:
df_llm = main_df.copy()

#for d in set(df.name):
#d = 'document-04.docx'
#preds = predictions[d]
#df_d = df_openai[df_openai['name'] == d].reset_index(drop=True)
#df_d['prediction'] = df_d.apply(lambda row: get_paragraph_predictions(row, predictions), axis=1)

def get_paragraph_predictions(row, predictions):
    preds = predictions.get(row['name'], [])
    return [
        p for p in preds
        if p['start_char'] >= row['start_char'] and p['end_char'] <= row['end_char']
    ]

df_llm['prediction'] = df_llm.apply(lambda row: get_paragraph_predictions(row, predictions), axis=1)


In [ ]:
df_llm[df_llm['name']!=df_llm['doc']]

In [ ]:
df_llm.head()

# Sanity check

In [ ]:
docs = list(set(df_llm.name))
docs.sort()

### Check number of paragraphs

In [ ]:
len_par_database = [{doc:len(doc_paragraphs[doc])} for doc in docs]
len_par_df = [{name: len(set(df_llm[df_llm['name']==name].paragraph_id))} for name in docs]
print(len_par_database)
print(len_par_df)


In [ ]:
print(len(set(doc_paragraphs['document-04.docx'])))
print(len(doc_paragraphs['document-04.docx']))

### Check all paragraphs in df

In [ ]:
def are_all_par(pars,df):
    return len(set(pars) - set(df.text_y.iloc[:])) == 0 

for d in docs:
    pars = doc_paragraphs[d]
    df = df_llm[df_llm['name']==d]
    if are_all_par(pars,df):
        print(f'All paragraphs in {d} are in df_llm')

# Save df_llm

In [ ]:
df_llm.to_csv('documents-02-08-sin05-conLLM-check.csv', index=False)